# Summarize One Model BT Tasks Results

This notebook loads one `experiment.json` from `results/` and outputs a table with:
- One overall row for the model
- One row per `task_type`

Metrics:
- **Task accuracy** = completed tasks / total tasks (and %)
- **Structure adherence** = valid trees / total trees (and %)

In [13]:
from pathlib import Path
import json
import pandas as pd

In [14]:
# Resolve repo root robustly (works even if notebook cwd is elsewhere)
def find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "src" / "tasks" / "tasks_100.json").exists() and (p / "results").exists():
            return p
    raise FileNotFoundError(
        "Could not find repo root. Start Jupyter from the project directory or set repo_root manually."
    )

repo_root = find_repo_root(Path.cwd())
results_root = repo_root / "results"

###### ! RESULTS TO BE DISPLAYED #####
# Model folder relative to `results/`.
# Example: wall_bt_2026-04-29_15-21-54/Qwen/Qwen2.5-3B-Instruct
model_rel = Path("wall_bt_2026-04-29_15-21-54/Qwen/Qwen2.5-3B-Instruct")
model_dir = model_rel if model_rel.is_absolute() else (results_root / model_rel)

main_results_path = model_dir / "main_results.json"
if not main_results_path.exists():
    raise FileNotFoundError(f"Missing main_results.json at: {main_results_path}")

main_results = json.loads(main_results_path.read_text(encoding="utf-8"))
all_tasks = main_results.get("all_tasks", [])
if not all_tasks:
    raise ValueError("No all_tasks found in main_results.json.")

model_name = main_results.get("model_id", model_dir.name)
print(f"Repo root: {repo_root}")
print(f"Model dir: {model_dir}")
print(f"Loaded model: {model_name}")
print(f"Using {len(all_tasks)} tasks from main_results.json")

Repo root: c:\Users\Owner\OneDrive\Documents\Work\Robotics Research\LLM_Structure_Adherence
Model dir: c:\Users\Owner\OneDrive\Documents\Work\Robotics Research\LLM_Structure_Adherence\results\wall_bt_2026-04-29_15-21-54\Qwen\Qwen2.5-3B-Instruct
Loaded model: Qwen/Qwen2.5-3B-Instruct
Using 100 tasks from main_results.json


In [15]:
import re

def ratio_str(numerator: int, denominator: int) -> str:
    if denominator == 0:
        return "0/0"
    return f"{numerator}/{denominator}"

def pct(numerator: int, denominator: int) -> float:
    if denominator == 0:
        return 0.0
    return (numerator / denominator) * 100.0

rows = []

# Overall row from the new main_results.json schema
overall_total_tasks = int(main_results.get("total_tasks", len(all_tasks)))
overall_completed_tasks = int(main_results.get("total_task_completion", 0))
overall_valid_trees = int(main_results.get("total_structure_adherence", 0))
overall_total_trees = sum(int(tr.get("bt_count", 0)) for tr in all_tasks)

rows.append(
    {
        "name": model_name,
        "task_accuracy": ratio_str(overall_completed_tasks, overall_total_tasks),
        "task_accuracy_pct": round(pct(overall_completed_tasks, overall_total_tasks), 2),
        "structure_adherence": ratio_str(overall_valid_trees, overall_total_trees),
        "structure_adherence_pct": round(pct(overall_valid_trees, overall_total_trees), 2),
    }
)

# Per-task-type rows from per-task files in model_dir/tasks/
tasks_dir = model_dir / "tasks"
if not tasks_dir.exists():
    raise FileNotFoundError(f"Missing tasks directory: {tasks_dir}")

task_files = sorted(tasks_dir.glob("task_*.json"))
if not task_files:
    raise ValueError(f"No task_*.json files found in: {tasks_dir}")

by_type: dict[str, dict[str, int]] = {}
pattern = re.compile(r"^task_(.+)_v\d+\.json$")

for task_file in task_files:
    m = pattern.match(task_file.name)
    task_type = m.group(1) if m else "unknown"

    task_data = json.loads(task_file.read_text(encoding="utf-8"))
    bucket = by_type.setdefault(
        task_type,
        {
            "total_tasks": 0,
            "completed_tasks": 0,
            "total_trees": 0,
            "valid_trees": 0,
        },
    )

    bucket["total_tasks"] += 1
    bucket["completed_tasks"] += 1 if task_data.get("task_completion", False) else 0
    bucket["total_trees"] += int(task_data.get("bt_count", 0))
    bucket["valid_trees"] += int(task_data.get("valid_structure_count", 0))

for task_type in sorted(by_type.keys()):
    stats = by_type[task_type]
    rows.append(
        {
            "name": task_type,
            "task_accuracy": ratio_str(stats["completed_tasks"], stats["total_tasks"]),
            "task_accuracy_pct": round(pct(stats["completed_tasks"], stats["total_tasks"]), 2),
            "structure_adherence": ratio_str(stats["valid_trees"], stats["total_trees"]),
            "structure_adherence_pct": round(pct(stats["valid_trees"], stats["total_trees"]), 2),
        }
    )

from IPython.display import display

df = pd.DataFrame(rows)

# Render percentage columns with % symbols
for col in ["task_accuracy_pct", "structure_adherence_pct"]:
    df[col] = df[col].map(lambda v: f"{float(v):.2f}%")

display(df)

# Print the first 15 completed tasks from all_tasks
completed_task_ids = [
    task.get("task_id")
    for task in all_tasks
    if task.get("task_completion", False)
]

print("\nFirst 15 completed task_ids:")
for task_id in completed_task_ids[:15]:
    print(task_id)

,name,task_accuracy,task_accuracy_pct,structure_adherence,structure_adherence_pct
0,Qwen/Qwen2.5-3B-Instruct,5/100,5.00%,370/483,76.60%
1,face_target,3/20,15.00%,83/88,94.32%
2,go_around_obstacle,0/20,0.00%,71/100,71.00%
3,go_to_multiple_targets,0/20,0.00%,76/100,76.00%
4,go_to_target,1/20,5.00%,77/98,78.57%
5,move_to_closest_target,1/20,5.00%,63/97,64.95%



First 15 completed task_ids:
go_to_target_v1
face_target_v2
face_target_v15
face_target_v16
move_to_closest_target_v4
